# 8. Churn Prediction Modelling

In [ ]:
X = model_df.drop(columns=['user_id','churn','days_since_last_trip'])
y = model_df['churn']

In [ ]:
# Because i added segment_name, i removed it from features
if 'segment_name' in model_df.columns:
    model_df = model_df.drop(columns=['segment_name'])

X = model_df.drop(columns=['user_id', 'churn', 'days_since_last_trip'])
y = model_df['churn']

X.dtypes

In [ ]:
X.select_dtypes(include='object')

In [ ]:
X.columns

In [ ]:
#Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Logistic Regression 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(X_train_scaled, y_train)

log_auc = roc_auc_score(y_test, log_reg.predict_proba(X_test_scaled)[:,1])
log_auc

In [ ]:
# Random Forest

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
rf_auc

# Model Performance

The Random Forest model achieved an AUC of 0.66, outperforming Logistic Regression (AUC = 0.64).

This indicates that:

Tree-based models capture non-linear behavioural patterns better.

The available features provide moderate predictive power.

Churn prediction in mobility data is inherently noisy due to irregular usage behaviour.

While performance is not extremely high, the model demonstrates meaningful separation between churned and active users.

## Feature Importance

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(10)

### Feature Importance Interpretation
Key Drivers of Churn

Top predictive features include:

Lower total_trips

Lower total_spend

Lower session engagement

Higher inactivity patterns

This suggests that:

Users with lower engagement and spending are more likely to churn.
Engagement behaviour (sessions, conversion rate) plays a role alongside ride frequency.

In [ ]:
# Added 2 behavioural ratios for improvement:
model_df['spend_per_trip'] = model_df['total_spend'] / (model_df['total_trips'] + 1)
model_df['sessions_per_trip'] = model_df['total_sessions'] / (model_df['total_trips'] + 1)

In [ ]:
#Retrain Random Forest but recreated X first to avoid leakage
X = model_df.drop(columns=['user_id','churn','days_since_last_trip'])
y = model_df['churn']

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
rf_auc

## Final Model interpretation 
### Feature Importance

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(10)

In [ ]:
len(X.columns), len(rf.feature_importances_)

In [ ]:
# Rebuild X and y (same as training)
X = model_df.drop(columns=['user_id', 'churn', 'days_since_last_trip'])
y = model_df['churn']

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

# Now feature importance will match X.columns
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(10)

### Key drivers of churn (Random Forest feature importance):

Value and frequency metrics (total_spend, total_trips) were the strongest signals of retention.

Customer type (segment) meaningfully differentiated churn risk.

Engagement and friction signals (avg_time_on_app, sessions_per_trip, avg_pages_visited) suggest that users who browse more may need better conversion support.

Pricing signals (avg_fare, avg_surge) indicate price sensitivity and surge exposure may contribute to churn.

### Recommendations:

Launch segment-specific retention: commuter loyalty + at-risk win-back offers.

Use engagement triggers: if sessions_per_trip is high, send “complete booking” nudges or targeted discounts.

Reduce surge-related churn with off-peak promotions or limited price-lock incentives.

In [ ]:
top10 = feature_importance.head(10).sort_values('importance')

plt.figure()
plt.barh(top10['feature'], top10['importance'])
plt.title("Top 10 Feature Importances — Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
model_df.groupby('segment')['churn'].mean()

Segment 0 — Lowest Churn (10.7%)
They are regular Commuters (High Value Users)

They have highest trips, Highest spend, Most recent activity, Lowest churn

Business Meaning:
These are the core revenue drivers.

Strategy:

Loyalty programmes, commuter subscriptions, referral rewards, premium features



Segment 1 — Medium Churn (16.5%)
They are engaged Value Seekers

They browse heavily, moderate trips, moderate spend, medium churn risk

Business Meaning:
They are active but possibly price-sensitive.

Strategy:
Targeted discounts, Smart push notifications, Personalised fare offers, Reduce booking friction

Segment 2 — Highest Churn (26.5%)
They are Occasional / At-Risk Users

They have Lowest trips, Highest recency, Lowest spend, Highest churn rate

Business Meaning:
This is the churn hotspot.

Strategy: Win-back campaigns, Time-limited offers, Email reactivation, Incentivised first ride back

### Segment-Level Churn Analysis

Segment 2 shows a churn rate of 26.5%, over twice the churn rate of Segment 0 (10.7%).

This indicates that customer segmentation meaningfully differentiates churn risk.

Retention strategies should prioritise Segment 2 users, while loyalty incentives should protect Segment 0 high-value commuters.